In [ ]:

import os
import gc
import asyncio
import json
import time
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from datasets import load_dataset, Audio
from jiwer import wer, cer
from tqdm.asyncio import tqdm_asyncio
from transformers import pipeline
from dotenv import load_dotenv

# LangChain
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq


In [ ]:
 
device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f"  GPU   : {gpu.name}")
    print(f"  VRAM  : {vram_gb:.1f} GB")
    print(f"  dtype : float16 (GPU)")
    if vram_gb < 4:
        print("  ⚠️  WARNING: <4 GB VRAM — distil-large-v3 needs ~3 GB minimum.")
        print("     Consider using 'openai/whisper-base' as a fallback.")
else:
    cpu_cores = os.cpu_count()
    print(f"  CPU cores : {cpu_cores}")
    print(f"  dtype     : float32 (CPU)")
    print(f"  ⚠️  CPU mode is ~10–20x slower than GPU for this model.")
 
print()


In [ ]:
 
# ─────────────────────────────────────────────
#  2. CACHE CONFIG  (adjust base_dir to yours)
# ─────────────────────────────────────────────
BASE_DIR = "D:/D/Dysarthria/ASR/huggingface_cache"
os.environ["HF_HOME"] = BASE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{BASE_DIR}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{BASE_DIR}/datasets"
os.environ["TORCH_HOME"] = f"{BASE_DIR}/torch"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
def transcribe(audio_array: np.ndarray, sampling_rate: int = 16000) -> dict:
    """
    Accept a numpy array directly (from HF dataset) instead of a file path.
    The pipeline accepts {"array": ..., "sampling_rate": ...} dicts natively.
    """
    audio_input = {"array": audio_array.astype(np.float32), "sampling_rate": sampling_rate}
    return asr_model(audio_input)
 
 
async def async_transcribe(audio_input) -> dict:
    """Async wrapper — offloads blocking PyTorch call to a thread."""
    return await asyncio.to_thread(transcribe, audio_input["array"], audio_input["sampling_rate"])
 
  
# ─────────────────────────────────────────────
#  5. PAUSE-AWARE FORMATTER
# ─────────────────────────────────────────────
def format_pause_aware_transcript(whisper_result: dict, pause_threshold: float = 0.8) -> dict:
    chunks = whisper_result.get("chunks", [])
    formatted_text = []
 
    for i, chunk in enumerate(chunks):
        word = chunk["text"].strip()
        if i > 0:
            prev_end = chunks[i - 1]["timestamp"][1]
            curr_start = chunk["timestamp"][0]
            if prev_end is not None and curr_start is not None:
                gap = curr_start - prev_end
                if gap >= pause_threshold:
                    formatted_text.append(f"[{gap:.1f}s pause]")
        formatted_text.append(word)
 
    return {"input": " ".join(formatted_text)}
 
 

In [ ]:
 
# ─────────────────────────────────────────────
#  6. FEW-SHOT PROMPT + LLM
# ─────────────────────────────────────────────
SYSTEM_PROMPT = """
You are an AI Speech Transcript Repair Assistant.
 
You receive automatic speech recognition transcripts from a speaker with a speech impairment.
 
Pauses marked like [2.0s pause] may indicate where short functional words 
(e.g., articles, auxiliary verbs, prepositions) were unintentionally omitted.
 
Your task is to minimally repair the sentence so it becomes grammatically correct 
while preserving the speaker's original wording, meaning, and intent.
 
Repair Rules:
1. Only insert or adjust small functional words when clearly necessary
2. Do NOT paraphrase, summarize, or restructure the sentence
3. Do NOT add new information or interpretations
4. If the sentence is already correct, return it unchanged
5. Ignore pause markers in the final output
6. Remove simple speech disfluencies:
   6.1 repeated whole words (e.g., "I I want" → "I want")
   6.2 filler sounds (e.g., "um", "uh")
7. Do NOT alter emphasis or stylistic repetition used intentionally
8. Preserve original sentence boundaries; do not merge or split sentences
 
Output Rules:
• Output exactly one corrected sentence
• No explanations, no commentary, no quotation marks
"""
 
examples = [
    {"input": "I [2.5s pause] go [1.2s pause] store [2.0s pause] milk",
     "output": "I am going to the store to get some milk."},
    {"input": "Want [3.0s pause] water [1.0s pause] cold",
     "output": "I want a glass of cold water."},
    {"input": "Turn [1.5s pause] light off [2.5s pause] room",
     "output": "Please turn off the lights in the room."},
    {"input": "My name [0.5s pause] is [4.0s pause] John",
     "output": "My name is John."},
]
 
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Transcript: {input}"),
    ("ai", "{output}"),
])
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
final_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    few_shot_prompt,
    ("human", "Transcript: {input}"),
])
 

In [ ]:
 
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0)
 
# ─────────────────────────────────────────────
#  7. FULL PIPELINE  (audio array → repaired text)
# ─────────────────────────────────────────────
transcribe_runnable = RunnableLambda(async_transcribe)
pause_formatter_runnable = RunnableLambda(format_pause_aware_transcript)
 
speech_repair_chain = (
    transcribe_runnable
    | pause_formatter_runnable
    | final_prompt
    | llm
    | StrOutputParser()
)
 

In [ ]:
# ─────────────────────────────────────────────
#  8. DATASET LOADER  — abnerh/TORGO-database
# ─────────────────────────────────────────────
def load_torgo(split: str = "test", max_samples: Optional[int] = 50):
    """
    Load the TORGO dataset from HuggingFace.
    Expected columns: audio, text (ground truth transcription)
    """
    print(f"Loading TORGO dataset (split='{split}', max={max_samples}) …")
    ds = load_dataset("abnerh/TORGO-database", split=split, trust_remote_code=True)
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
 
    if max_samples and max_samples < len(ds):
        ds = ds.select(range(max_samples))
 
    print(f"  {len(ds)} samples loaded.\n")
    return ds
 
 